In [0]:
# Packages required by all code.
# Versions of Databricks code are not locked since Databricks ensures changes are backwards compatible.
# Versions of open source packages are locked since package authors often make backwards compatible changes
%pip install -qqqq -U \
  databricks-vectorsearch databricks-agents pydantic databricks-sdk mlflow mlflow-skinny `# For agent & data pipeline code` \
  pypdf==4.1.0  `# PDF parsing` \
  markdownify==0.12.1  `# HTML parsing` \
  pypandoc_binary==1.13  `# DOCX parsing` \
  transformers==4.41.1 torch==2.3.0 tiktoken==0.7.0 langchain-text-splitters==0.2.0. `# get_recursive_character_text_splitter` \

# Restart to load the packages into the Python environment
dbutils.library.restartPython()

In [0]:
%run ./00_config

In [0]:
import mlflow

mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

## Start MLflow run for tracking

In [0]:
mlflow.start_run(run_name=POC_DATA_PIPELINE_RUN_NAME)

## Prepare data for creating embeddings and the vector search index

### Chunk data



In [0]:
from typing import Literal, Optional, Any, Callable
from databricks.vector_search.client import VectorSearchClient
from pyspark.sql.functions import explode
import pyspark.sql.functions as func
from typing import Callable
from langchain_text_splitters import RecursiveCharacterTextSplitter
from transformers import AutoTokenizer
import tiktoken
from pyspark.sql.types import StructType, StringType, StructField, MapType, ArrayType

In [0]:
chunk_size = 368

In [0]:
from pyspark.sql import functions as func  
from pyspark.sql import SparkSession  
from transformers import AutoTokenizer  
  
# Initialize the tokenizer (replace 'bert-base-uncased' with your model name)  
tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-large-en-v1.5")
  
def truncate_column_by_tokens(docs_table: str, doc_column: str, chunk_size: int, chunked_docs_table: str = None) -> str:  
    # Default the chunked_docs_table name if not provided  
    chunked_docs_table = chunked_docs_table or f"{docs_table}_truncated"  
    
    print(f"Truncating content in `{doc_column}` of table `{docs_table}` to {chunk_size} tokens...")  
  
    # Read the original documents table  
    raw_docs = spark.read.table(docs_table)  
  
    # Create a new DataFrame with tokenized content  
    tokenized_docs = raw_docs.select(  
        "*",  
        func.expr(f"size(split({doc_column}, ' ')) as token_count")  # Count tokens based on space  
    )  
  
    # Truncate the specified column based on the provided token size  
    def truncate_to_tokens(text):  
        if text is None:  
            return None  
        tokens = tokenizer.tokenize(text)  
        return ' '.join(tokens[:chunk_size])  
  
    # Apply the truncation logic to each row  
    truncated_docs = tokenized_docs.rdd.map(lambda row: (  
        row[0],  # assuming the first element is some ID or unique identifier  
        truncate_to_tokens(row[1]),  # truncating the specified column  
        *row[2:]  # include other columns as is  
    )).toDF(["id", "truncated_content", *tokenized_docs.columns[2:]])  # Adjust column names as needed  
  
    # Optional: Add a primary key for each truncated document  
    chunks_with_ids = truncated_docs.withColumn(  
        "chunk_id",   
        func.md5(func.col("truncated_content"))  
    )  
  
    # Count the number of chunks created  
    print(f"Created {chunks_with_ids.count()} truncated documents!")  
  
    # Write to Delta Table  
    chunks_with_ids.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(chunked_docs_table)  
  
    return chunked_docs_table  
  


In [0]:
TABLE_NAME = "wikipedia"
INPUT_DELTA_TABLE = f"{UC_CATALOG}.{UC_SCHEMA}.{TABLE_NAME}"
CHUNKED_DELTA_TABLE = f"{UC_CATALOG}.{UC_SCHEMA}.wikipedia_chunked"
VECTOR_INDEX_NAME = f"{UC_CATALOG}.{UC_SCHEMA}.{TABLE_NAME}_vector_index"


In [0]:
spark.read.table(INPUT_DELTA_TABLE).printSchema()

In [0]:
from pyspark.sql import functions as func  
from pyspark.sql import SparkSession  
from transformers import AutoTokenizer  
  
# Initialize the tokenizer (replace 'BAAI/bge-large-en-v1.5' with your model name)  
tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-large-en-v1.5")  
  
def truncate_column_by_tokens(docs_table: str, doc_column: str, chunk_size: int, chunked_docs_table: str = None) -> str:  
    # Default the chunked_docs_table name if not provided  
    chunked_docs_table = chunked_docs_table or f"{docs_table}_truncated"  
  
    print(f"Truncating content in `{doc_column}` of table `{docs_table}` to {chunk_size} tokens...")  
  
    # Read the original documents table  
    raw_docs = spark.read.table(docs_table)  
  
    # Create a new DataFrame with tokenized content  
    tokenized_docs = raw_docs.select(  
        "*",   
        func.expr(f"size(split({doc_column}, ' ')) as token_count")  # Count tokens based on spaces  
    )  
  
    # Truncate the specified column based on the provided token size  
    def truncate_to_tokens(text):  
        if text is None:  
            return None  
        tokens = tokenizer.tokenize(text)  
        return ' '.join(tokens[:chunk_size])  
  
    # Apply the truncation logic to each row  
    truncated_docs = tokenized_docs.rdd.map(lambda row: {  
        **row.asDict(),  # Include all original columns as a dictionary  
        "truncated_content": truncate_to_tokens(row[doc_column])  # Add the truncated content  
    })  
  
    # Create the new DataFrame with a new column name for the truncated content  
    truncated_df = spark.createDataFrame(truncated_docs)  
  
    # Optional: Add a primary key for each truncated document  
    chunks_with_ids = truncated_df.withColumn(  
        "chunk_id",  
        func.md5(func.col("truncated_content"))  
    )  
  
    # Count the number of chunks created  
    print(f"Created {chunks_with_ids.count()} truncated documents!")  
  
    # Write to Delta Table  
    chunks_with_ids.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(chunked_docs_table)  
  
    return chunked_docs_table  

In [0]:
truncated_table = truncate_column_by_tokens(INPUT_DELTA_TABLE, "Plot", chunk_size, CHUNKED_DELTA_TABLE)  

In [0]:
spark.sql(f"ALTER TABLE {CHUNKED_DELTA_TABLE} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")

## Create the vector search index

In [0]:
from databricks.vector_search.client import VectorSearchClient


In [0]:
client = VectorSearchClient()

index = client.create_delta_sync_index(
  endpoint_name=VECTOR_SEARCH_ENDPOINT_NAME,
  source_table_name=CHUNKED_DELTA_TABLE,
  index_name=VECTOR_INDEX_NAME,
  pipeline_type="TRIGGERED",
  primary_key="id",
  embedding_source_column="truncated_content",
  embedding_model_endpoint_name=EMBEDDING_MODEL_NAME
)

In [0]:
mlflow.end_run()